In [1]:
print("Jay Ganesh")

Jay Ganesh


In [2]:
import os

In [3]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/Mehta-Kartik/FirstMLOPsProject.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="Mehta-Kartik"
os.environ["MLFLOW_TRACKING_PASSWORD"]="2a74092db8949c3d1feffbb9841705c2513185bf"

In [ ]:
import os
# %pwd

'd:\\ProjectAarya\\MLOPs\\Code\\Ch18 FirstMLOPs\\FirstMLOPsProject\\research'

In [ ]:
# os.chdir("../")
# %pwd

'd:\\ProjectAarya\\MLOPs\\Code\\Ch18 FirstMLOPs\\FirstMLOPsProject'

In [9]:
## Setup Entity
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str


In [8]:
from src.First_End_to_End_Project.constants import *
from src.First_End_to_End_Project.utils.common import read_yaml,create_directories,save_json

In [11]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):   
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath) 
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_model_evaluation_config(self)->ModelEvaluationConfig:
        config=self.config.model_evaluation
        param=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config=ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=param,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri=os.getenv("MLFLOW_TRACKING_URI"),
        )

        return model_evaluation_config

In [12]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [13]:
class ModelEvaluation:
    def __init__(self,config:ModelEvaluationConfig):
        self.config=config
    
    def eval_metrics(self,actual,predict):
        rmse=np.sqrt(mean_squared_error(actual,predict))
        mae=mean_absolute_error(actual,predict)
        r2=r2_score(actual,predict)
        return rmse,mae,r2
    
    def log_into_mlflow(self):
        test_data=pd.read_csv(self.config.test_data_path)
        model=joblib.load(self.config.model_path)

        test_x=test_data.drop([self.config.target_column],axis=1)
        test_y=test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store=urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities=model.predict(test_x)
            (rmse,mae,r2)=self.eval_metrics(actual=test_y,predict=predicted_qualities)

            #Save the metrics as local
            score={"rmse":rmse,"mae":mae,"r2":r2}
            save_json(path=Path(self.config.metric_file_name),data=score)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse",rmse)
            mlflow.log_metric("mae",mae)
            mlflow.log_metric("r2",r2)


            #If model Registry does not work with file store
            if tracking_url_type_store!="file":
                mlflow.sklearn.log_model(model,"Model",registered_model_name="ElasticNetModel")
            else:
                mlflow.sklearn.log_model(model,"Model")
        

In [14]:
try:
    config=ConfigurationManager()
    model_eval_config=config.get_model_evaluation_config()
    model_eval_config=ModelEvaluation(config=model_eval_config)
    model_eval_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-03-18 18:39:56,744:INFO:common:yaml file: D:\ProjectAarya\MLOPs\Code\Ch18 FirstMLOPs\FirstMLOPsProject\config\config.yaml loaded successfully]
[2026-03-18 18:39:56,774:INFO:common:yaml file: D:\ProjectAarya\MLOPs\Code\Ch18 FirstMLOPs\FirstMLOPsProject\params.yaml loaded successfully]
[2026-03-18 18:39:56,787:INFO:common:yaml file: D:\ProjectAarya\MLOPs\Code\Ch18 FirstMLOPs\FirstMLOPsProject\schema.yaml loaded successfully]
[2026-03-18 18:39:56,790:INFO:common:created directory at: artifacts]
[2026-03-18 18:39:56,795:INFO:common:created directory at: artifacts/model_evaluation]
[2026-03-18 18:40:01,344:INFO:common:Json file saved at: artifacts\model_evaluation\metrics.json]


d:\ProjectAarya\MLOPs\Code\Ch18 FirstMLOPs\FirstMLOPsProject\venv1\lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Successfully registered model 'ElasticNetModel'.
2026/03/18 18:40:38 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: ElasticNetModel, version 1
Created version '1' of model 'ElasticNetModel'.
